# Augmentation – Day 4: Evaluation

**Student Name:** Traver Dinten
**Country:** Switzerland
**Semester term:** FS26

## Evaluation Approach Definition

The selected metric is **SSIM (Structural Similarity Index)**, which quantifies the perceived structural similarity between the augmented image and the original baseline image. SSIM is specifically chosen because the problem statement requires accurate assessment of whether structural features (ice-rock boundaries, surface textures) are preserved after augmentation, and SSIM directly reflects changes in luminance, contrast, and structural information. The metric ranges from 0 to 1, where 1 indicates perfect structural similarity. The validity of this metric depends on having a well-defined reference image (the original), which is the case here since augmentation starts from the unmodified Sentinel-2 satellite composite.

## Setup and Data Loading

In [ ]:
import sys, os
sys.path.insert(0, os.path.join("..", "helpers"))

import numpy as np
import cv2
import matplotlib.pyplot as plt
from skimage.metrics import structural_similarity as ssim
from gee_utils import load_image

DATA_DIR = os.path.join("..", "data", "raw")
FIGURES_DIR = os.path.join("..", "figures")
os.makedirs(FIGURES_DIR, exist_ok=True)

IMAGE_PATH = os.path.join(DATA_DIR, "aletsch_jungfraufirn_s2.tif")
img_rgb = load_image(IMAGE_PATH)
print(f"Image shape: {img_rgb.shape}")

In [ ]:
GAMMA_VALUES = [0.5, 0.75, 1.0, 1.5, 2.0]
NOISE_SIGMAS = [0.0, 0.02, 0.05, 0.10]

img_norm = img_rgb.astype(np.float64) / 255.0
np.random.seed(42)

def apply_gamma(image, gamma):
    return np.clip(np.power(image, gamma), 0, 1)

def add_gaussian_noise(image, sigma):
    if sigma == 0:
        return image.copy()
    noise = np.random.normal(0, sigma, image.shape)
    return np.clip(image + noise, 0, 1)

## Evaluation: Gamma Correction

In [ ]:
baseline = img_norm.copy()
gamma_ssim = {}

for gamma in GAMMA_VALUES:
    augmented = apply_gamma(img_norm, gamma)
    score = ssim(baseline, augmented, channel_axis=2, data_range=1.0)
    gamma_ssim[gamma] = score

baseline_ssim = gamma_ssim[1.0]

print("Gamma Correction – SSIM Evaluation (Sentinel-2)")
print("-" * 55)
print(f"{'Configuration':<25} {'SSIM':>8} {'Δ Baseline (%)':>16}")
print("-" * 55)
for gamma, score in gamma_ssim.items():
    label = "Baseline (γ=1.0)" if gamma == 1.0 else f"γ={gamma}"
    rel_change = ((score - baseline_ssim) / baseline_ssim) * 100 if baseline_ssim != 0 else 0
    print(f"{label:<25} {score:>8.4f} {rel_change:>15.1f}%")

## Evaluation: Gaussian Noise

In [ ]:
noise_ssim = {}

for sigma in NOISE_SIGMAS:
    augmented = add_gaussian_noise(img_norm, sigma)
    score = ssim(baseline, augmented, channel_axis=2, data_range=1.0)
    noise_ssim[sigma] = score

noise_baseline = noise_ssim[0.0]

print("Gaussian Noise – SSIM Evaluation (Sentinel-2)")
print("-" * 55)
print(f"{'Configuration':<25} {'SSIM':>8} {'Δ Baseline (%)':>16}")
print("-" * 55)
for sigma, score in noise_ssim.items():
    label = "Baseline (σ=0)" if sigma == 0 else f"σ={sigma}"
    rel_change = ((score - noise_baseline) / noise_baseline) * 100 if noise_baseline != 0 else 0
    print(f"{label:<25} {score:>8.4f} {rel_change:>15.1f}%")

## Comparative Visualization

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

gammas = list(gamma_ssim.keys())
gamma_scores = list(gamma_ssim.values())
colors_g = ["#2ecc71" if g == 1.0 else "#3498db" for g in gammas]
axes[0].bar([f"γ={g}" for g in gammas], gamma_scores, color=colors_g, edgecolor="black", linewidth=0.5)
axes[0].set_ylabel("SSIM")
axes[0].set_title("SSIM vs. Gamma Correction (Sentinel-2)")
axes[0].set_ylim(0, 1.05)
axes[0].axhline(y=baseline_ssim, color="gray", linestyle="--", alpha=0.5)

sigmas = list(noise_ssim.keys())
noise_scores = list(noise_ssim.values())
colors_n = ["#2ecc71" if s == 0.0 else "#e74c3c" for s in sigmas]
axes[1].bar([f"σ={s}" for s in sigmas], noise_scores, color=colors_n, edgecolor="black", linewidth=0.5)
axes[1].set_ylabel("SSIM")
axes[1].set_title("SSIM vs. Gaussian Noise (Sentinel-2)")
axes[1].set_ylim(0, 1.05)
axes[1].axhline(y=noise_baseline, color="gray", linestyle="--", alpha=0.5)

plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, "day04_ssim_comparison.png"), dpi=150, bbox_inches="tight")
plt.show()